# Amazon ML Challenge 2026 — Production Inference & Submission

Production test-inference notebook after blocking.

Locked baseline:
- V2 normalization
- token-frequency threshold = `1000`
- top-K rare tokens = `2`
- LightGBM matcher with the current 13 features
- current experimental decision threshold = `0.998`

`0.998` is configurable and should be replaced if the corrected validation experiment finds a better threshold.

This notebook is chunked and restartable for a 16 GB RAM machine.


In [1]:
from pathlib import Path
import re
import gc
import json
import math
import pickle
import time
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio, WRatio

# ============================================================
# PROJECT ROOT
# ============================================================
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = ROOT / "models"

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_DIR = OUTPUT_DIR / "inference_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# Locked challenge configuration
TOKEN_FREQ_THRESHOLD = 1000
TOP_K = 2
MATCH_THRESHOLD = 0.998

# Memory-safe inference
S1_CHUNK_SIZE = 10_000
INFERENCE_BATCH_SIZE = 100

MODEL_PATH = MODEL_DIR / "entity_matcher_lightgbm.pkl"
MODEL_CONFIG_PATH = MODEL_DIR / "matcher_config.pkl"
RESULT_PATH = OUTPUT_DIR / "matching_results.tsv"

print("=" * 70)
print("AMAZON ML CHALLENGE 2026 — MEMORY-SAFE PRODUCTION INFERENCE")
print("=" * 70)
print("ROOT:", ROOT)
print("RAW:", RAW_DIR)
print("MODEL:", MODEL_PATH)
print("CONFIG:", MODEL_CONFIG_PATH)
print("OUTPUT:", OUTPUT_DIR)
print("CHUNKS:", CHUNK_DIR)
print("THRESHOLD:", MATCH_THRESHOLD)
print("S1 CHUNK:", S1_CHUNK_SIZE)
print("INFERENCE BATCH:", INFERENCE_BATCH_SIZE)
print("=" * 70)


AMAZON ML CHALLENGE 2026 — MEMORY-SAFE PRODUCTION INFERENCE
ROOT: d:\amzn_ml_challenge
RAW: d:\amzn_ml_challenge\data\raw
MODEL: d:\amzn_ml_challenge\models\entity_matcher_lightgbm.pkl
CONFIG: d:\amzn_ml_challenge\models\matcher_config.pkl
OUTPUT: d:\amzn_ml_challenge\output
CHUNKS: d:\amzn_ml_challenge\output\inference_chunks
THRESHOLD: 0.998
S1 CHUNK: 10000
INFERENCE BATCH: 100


In [2]:
print("ROOT EXISTS:", ROOT.exists())
print("RAW EXISTS:", RAW_DIR.exists())
print("MODEL EXISTS:", MODEL_DIR.exists())

print("\nRaw files:")
for f in sorted(RAW_DIR.glob("*.tsv")):
    print(" ", f.name)

print("\nModel files:")
for f in sorted(MODEL_DIR.glob("*")):
    print(" ", f.name)

ROOT EXISTS: True
RAW EXISTS: True
MODEL EXISTS: True

Raw files:
  test_source1.tsv
  test_source2.tsv
  test_source3.tsv
  train_ground_truth.tsv
  train_source1.tsv
  train_source2.tsv
  train_source3.tsv

Model files:
  entity_matcher_lightgbm.pkl
  matcher_config.pkl


## 1. Load matcher model

In [3]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Matcher model not found: {MODEL_PATH}. "
        "Run the matcher-training notebook first."
    )

# Load LightGBM matcher
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print("Loaded model:", type(model).__name__)


# Load matcher configuration
if not MODEL_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Matcher config not found: {MODEL_CONFIG_PATH}"
    )

with open(MODEL_CONFIG_PATH, "rb") as f:
    matcher_config = pickle.load(f)

print("Loaded config:", type(matcher_config).__name__)

print("\nMatcher configuration:")
print(matcher_config)

Loaded model: LGBMClassifier
Loaded config: dict

Matcher configuration:
{'random_state': 42, 'blocking': {'normalization': 'V2', 'token_frequency_threshold': 1000, 'top_k': 2}, 'matcher_features': ['country_same', 'name_exact', 'address_exact', 'name_length_diff', 'address_length_diff', 'name_token_jaccard', 'name_token_containment', 'address_token_jaccard', 'address_token_containment', 'name_char_ratio', 'address_char_ratio', 'name_wratio', 'address_wratio'], 'recommended_threshold_from_current_10k_experiment': 0.998}


## 2. Load test data

In [4]:

test_s1 = pd.read_csv(RAW_DIR / "test_source1.tsv", sep="\t", dtype=str)
test_s2 = pd.read_csv(RAW_DIR / "test_source2.tsv", sep="\t", dtype=str)
test_s3 = pd.read_csv(RAW_DIR / "test_source3.tsv", sep="\t", dtype=str)

required = {"entity_id", "business_name", "business_address", "country"}

for name, df in [("S1", test_s1), ("S2", test_s2), ("S3", test_s3)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing: {missing}")
    if df["entity_id"].duplicated().any():
        raise ValueError(f"{name} has duplicate entity_id values")

print("S1:", test_s1.shape)
print("S2:", test_s2.shape)
print("S3:", test_s3.shape)


S1: (1732544, 4)
S2: (4887273, 4)
S3: (5082316, 4)


## 3. Locked V2 normalization

In [5]:

import unicodedata

def normalize_name_v2(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_address_v2(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    replacements = {
        r"\bave\b":"avenue", r"\bav\b":"avenue",
        r"\bst\b":"street", r"\brd\b":"road", r"\bdr\b":"drive",
        r"\bln\b":"lane", r"\btrl\b":"trail", r"\bhwy\b":"highway",
        r"\bblvd\b":"boulevard", r"\bpkwy\b":"parkway",
        r"\bct\b":"court", r"\bpl\b":"place", r"\bcir\b":"circle",
        r"\bter\b":"terrace", r"\bste\b":"suite", r"\bapt\b":"apartment",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

for df in (test_s1, test_s2, test_s3):
    df["normalized_name_v2"] = df["business_name"].map(normalize_name_v2)
    df["normalized_address_v2"] = df["business_address"].map(normalize_address_v2)

print("Normalization complete.")


Normalization complete.


## 4. Build locked V2 indexes

Rarity follows the locked blocker: frequencies are computed from Source 2 and the resulting rare tokens are used against both S2 and S3 indexes.


In [6]:

def build_exact_index(df, column):
    idx = defaultdict(list)
    for value, entity_id in zip(df[column].to_numpy(), df["entity_id"].to_numpy()):
        if value:
            idx[value].append(entity_id)
    return idx

def build_token_index(df, column):
    idx = defaultdict(list)
    freq = Counter()
    values = df[column].to_numpy()
    ids = df["entity_id"].to_numpy()

    for value in values:
        if value:
            freq.update(set(value.split()))

    for value, entity_id in zip(values, ids):
        if value:
            for token in set(value.split()):
                idx[token].append(entity_id)
    return idx, freq

s2_name_exact = build_exact_index(test_s2, "normalized_name_v2")
s2_address_exact = build_exact_index(test_s2, "normalized_address_v2")
s3_name_exact = build_exact_index(test_s3, "normalized_name_v2")
s3_address_exact = build_exact_index(test_s3, "normalized_address_v2")

s2_name_index, s2_name_freq = build_token_index(test_s2, "normalized_name_v2")
s2_address_index, s2_address_freq = build_token_index(test_s2, "normalized_address_v2")
s3_name_index, _ = build_token_index(test_s3, "normalized_name_v2")
s3_address_index, _ = build_token_index(test_s3, "normalized_address_v2")

print("Indexes built.")
print("S2 name tokens:", len(s2_name_index))
print("S2 address tokens:", len(s2_address_index))
print("S3 name tokens:", len(s3_name_index))
print("S3 address tokens:", len(s3_address_index))
gc.collect()


Indexes built.
S2 name tokens: 669307
S2 address tokens: 604075
S3 name tokens: 712682
S3 address tokens: 575653


0

## 5. Locked blocker

In [7]:

def generate_v2_block_candidates_1000(s1_row, top_k=TOP_K):
    name = s1_row["normalized_name_v2"]
    address = s1_row["normalized_address_v2"]

    s2_candidates = set()
    s3_candidates = set()

    if name:
        s2_candidates.update(s2_name_exact.get(name, []))
        s3_candidates.update(s3_name_exact.get(name, []))

    if address:
        s2_candidates.update(s2_address_exact.get(address, []))
        s3_candidates.update(s3_address_exact.get(address, []))

    if name:
        eligible = [
            t for t in set(name.split())
            if s2_name_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD
        ]
        for token in sorted(
            eligible, key=lambda t: s2_name_freq.get(t, 10**9)
        )[:top_k]:
            s2_candidates.update(s2_name_index.get(token, []))
            s3_candidates.update(s3_name_index.get(token, []))

    if address:
        eligible = [
            t for t in set(address.split())
            if s2_address_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD
        ]
        for token in sorted(
            eligible, key=lambda t: s2_address_freq.get(t, 10**9)
        )[:top_k]:
            s2_candidates.update(s2_address_index.get(token, []))
            s3_candidates.update(s3_address_index.get(token, []))

    return list(s2_candidates) + list(s3_candidates)

sample_candidates = generate_v2_block_candidates_1000(test_s1.iloc[0])
print("Smoke candidate count:", len(sample_candidates))


Smoke candidate count: 610


## 6. Target lookup frames

In [8]:
def target_frame(df):

    return df[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "normalized_name_v2",
            "normalized_address_v2",
        ]
    ].rename(columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_name_v2",
        "normalized_address_v2": "target_address_v2",
    })


s2_target = target_frame(test_s2)
s3_target = target_frame(test_s3)

print("S2 target rows:", len(s2_target))
print("S3 target rows:", len(s3_target))

S2 target rows: 4887273
S3 target rows: 5082316


## 7. Matcher features

In [9]:

def token_jaccard(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    union = sa | sb
    return len(sa & sb) / len(union) if union else 0.0

def token_containment(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    m = min(len(sa), len(sb))
    return len(sa & sb) / m if m else 0.0

FEATURE_COLS = [
    "country_same","name_exact","address_exact",
    "name_length_diff","address_length_diff",
    "name_token_jaccard","name_token_containment",
    "address_token_jaccard","address_token_containment",
    "name_char_ratio","address_char_ratio",
    "name_wratio","address_wratio",
]

def build_feature_frame(df):
    sn = df["s1_name_v2"].fillna("")
    sa = df["s1_address_v2"].fillna("")
    tn = df["target_name_v2"].fillna("")
    ta = df["target_address_v2"].fillna("")

    f = pd.DataFrame(index=df.index)
    f["country_same"] = (
        df["s1_country"].fillna("") == df["target_country"].fillna("")
    ).astype("int8")
    f["name_exact"] = (sn == tn).astype("int8")
    f["address_exact"] = (sa == ta).astype("int8")
    f["name_length_diff"] = (sn.str.len() - tn.str.len()).abs().astype("float32")
    f["address_length_diff"] = (sa.str.len() - ta.str.len()).abs().astype("float32")

    f["name_token_jaccard"] = [token_jaccard(a,b) for a,b in zip(sn,tn)]
    f["name_token_containment"] = [token_containment(a,b) for a,b in zip(sn,tn)]
    f["address_token_jaccard"] = [token_jaccard(a,b) for a,b in zip(sa,ta)]
    f["address_token_containment"] = [token_containment(a,b) for a,b in zip(sa,ta)]

    f["name_char_ratio"] = [ratio(a,b)/100 for a,b in zip(sn,tn)]
    f["address_char_ratio"] = [ratio(a,b)/100 for a,b in zip(sa,ta)]
    f["name_wratio"] = [WRatio(a,b)/100 for a,b in zip(sn,tn)]
    f["address_wratio"] = [WRatio(a,b)/100 for a,b in zip(sa,ta)]
    return f


## 8. Chunk inference

Only accepted pairs are returned from each chunk. Candidate matrices are discarded before moving to the next chunk.


In [12]:
def infer_chunk(s1_chunk):
    """
    Process one outer S1 chunk in small memory-safe batches.
    Only accepted predictions are retained.
    """
    outputs = []
    n_batches = math.ceil(len(s1_chunk) / INFERENCE_BATCH_SIZE)

    for batch_idx in range(n_batches):
        start = batch_idx * INFERENCE_BATCH_SIZE
        end = min(start + INFERENCE_BATCH_SIZE, len(s1_chunk))
        batch = s1_chunk.iloc[start:end].copy()

        rows = []

        for _, row in batch.iterrows():

            candidates = generate_v2_block_candidates_1000(
                row,
                top_k=TOP_K
            )

            for target_id in candidates:
                rows.append(
                    (row["entity_id"], target_id)
                )

        if not rows:
            del batch, rows
            gc.collect()
            continue

        candidate_pairs = pd.DataFrame(
            rows,
            columns=["source1_entity_id", "target_entity_id"]
        )
        del rows

        # Attach S1 fields.
        s1_fields = batch[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "normalized_name_v2",
                "normalized_address_v2",
            ]
        ].rename(columns={
            "entity_id": "source1_entity_id",
            "business_name": "source1_business_name",
            "business_address": "source1_business_address",
            "country": "source1_country",
            "normalized_name_v2": "source1_name_v2",
            "normalized_address_v2": "source1_address_v2",
        })

        candidate_features = candidate_pairs.merge(
            s1_fields,
            on="source1_entity_id",
            how="left",
            sort=False,
        )
        del candidate_pairs, s1_fields

        # Split target source by ID prefix.
        s2_mask = candidate_features["target_entity_id"].str.startswith("S2-")
        s2_candidates = candidate_features.loc[s2_mask].copy()
        s3_candidates = candidate_features.loc[~s2_mask].copy()
        del candidate_features

        if not s2_candidates.empty:
            s2_candidates = s2_candidates.merge(
                s2_target,
                on="target_entity_id",
                how="left",
                sort=False,
            )

        if not s3_candidates.empty:
            s3_candidates = s3_candidates.merge(
                s3_target,
                on="target_entity_id",
                how="left",
                sort=False,
            )

        parts = []
        if not s2_candidates.empty:
            parts.append(s2_candidates)
        if not s3_candidates.empty:
            parts.append(s3_candidates)

        if not parts:
            del batch
            gc.collect()
            continue

        candidate_features = pd.concat(parts, ignore_index=True)
        del s2_candidates, s3_candidates, parts

        # Build the exact 13 model features.
        sn = candidate_features["source1_name_v2"].fillna("")
        sa = candidate_features["source1_address_v2"].fillna("")
        tn = candidate_features["target_name_v2"].fillna("")
        ta = candidate_features["target_address_v2"].fillna("")

        f = pd.DataFrame(index=candidate_features.index)

        f["country_same"] = (
            candidate_features["source1_country"].fillna("")
            == candidate_features["target_country"].fillna("")
        ).astype("int8")

        f["name_exact"] = (sn == tn).astype("int8")
        f["address_exact"] = (sa == ta).astype("int8")

        f["name_length_diff"] = (
            sn.str.len() - tn.str.len()
        ).abs().astype("float32")

        f["address_length_diff"] = (
            sa.str.len() - ta.str.len()
        ).abs().astype("float32")

        f["name_token_jaccard"] = [
            token_jaccard(a, b) for a, b in zip(sn, tn)
        ]
        f["name_token_containment"] = [
            token_containment(a, b) for a, b in zip(sn, tn)
        ]
        f["address_token_jaccard"] = [
            token_jaccard(a, b) for a, b in zip(sa, ta)
        ]
        f["address_token_containment"] = [
            token_containment(a, b) for a, b in zip(sa, ta)
        ]

        f["name_char_ratio"] = [
            ratio(a, b) / 100.0 for a, b in zip(sn, tn)
        ]
        f["address_char_ratio"] = [
            ratio(a, b) / 100.0 for a, b in zip(sa, ta)
        ]
        f["name_wratio"] = [
            WRatio(a, b) / 100.0 for a, b in zip(sn, tn)
        ]
        f["address_wratio"] = [
            WRatio(a, b) / 100.0 for a, b in zip(sa, ta)
        ]

        probabilities = model.predict_proba(f[FEATURE_COLS])[:, 1]

        accepted_mask = probabilities >= MATCH_THRESHOLD

        if accepted_mask.any():
            accepted = candidate_features.loc[
                accepted_mask,
                ["source1_entity_id", "target_entity_id"]
            ].copy()
            accepted["match_probability"] = probabilities[accepted_mask]
            outputs.append(accepted)

        print(
            f"    batch {batch_idx + 1:03d}/{n_batches:03d} | "
            f"candidates={len(candidate_features):,} | "
            f"accepted={int(accepted_mask.sum()):,}"
        )

        del f, candidate_features, probabilities, batch
        gc.collect()

    if not outputs:
        return pd.DataFrame(
            columns=[
                "source1_entity_id",
                "target_entity_id",
                "match_probability",
            ]
        )

    return pd.concat(outputs, ignore_index=True)


## 9. Smoke test — run this before full inference

In [13]:

smoke = infer_chunk(test_s1.iloc[:100].copy())
print("Smoke accepted pairs:", len(smoke))
print(smoke.head(20))


    batch 001/001 | candidates=200,026 | accepted=277
Smoke accepted pairs: 277
   source1_entity_id target_entity_id  match_probability
0       S1-156285671     S2-971574924           0.999499
1       S1-156285671     S2-180394702           0.999153
2       S1-689823050      S2-65478038           0.999865
3       S1-689823050     S2-319486263           0.998960
4       S1-689823050     S2-212601842           0.998883
5       S1-689823050     S2-829077756           0.999547
6       S1-921369899     S2-107533354           0.999626
7       S1-921369899     S2-995426671           0.999170
8       S1-921369899     S2-808920974           0.999911
9       S1-481669221     S2-284602321           0.999896
10      S1-909865979     S2-528901108           0.999924
11      S1-742053041      S2-73207920           0.999924
12      S1-742053041     S2-317425677           0.999939
13      S1-742053041     S2-315673501           0.998985
14      S1-742053041     S2-446206310           0.998953
15      

## 10. Full test inference

**Restartable:** existing completed chunk files are skipped.

If the IDE/kernel crashes:
1. restart kernel
2. rerun through Cell 9
3. leave `CLEAR_OLD_CHUNKS = False`
4. rerun this cell


In [14]:
# ============================================================
# FULL PRODUCTION INFERENCE
# Restartable: existing completed chunks are skipped.
# ============================================================

CLEAR_OLD_CHUNKS = False

if CLEAR_OLD_CHUNKS:
    for p in CHUNK_DIR.glob("matches_*.tsv"):
        p.unlink()

n_chunks = math.ceil(len(test_s1) / S1_CHUNK_SIZE)

print("S1 rows:", f"{len(test_s1):,}")
print("Outer chunks:", n_chunks)
print("Inner batch size:", INFERENCE_BATCH_SIZE)
print()

for chunk_idx in range(n_chunks):
    path = CHUNK_DIR / f"matches_{chunk_idx:05d}.tsv"

    if path.exists():
        print(f"[{chunk_idx + 1}/{n_chunks}] SKIP {path.name}")
        continue

    start = chunk_idx * S1_CHUNK_SIZE
    end = min(start + S1_CHUNK_SIZE, len(test_s1))

    chunk = test_s1.iloc[start:end].copy()

    t0 = time.time()

    result = infer_chunk(chunk)

    result.to_csv(
        path,
        sep="\t",
        index=False
    )

    print(
        f"[{chunk_idx + 1}/{n_chunks}] "
        f"S1={len(chunk):,} "
        f"accepted={len(result):,} "
        f"time={time.time() - t0:.1f}s"
    )

    del chunk, result
    gc.collect()

print("\nALL INFERENCE CHUNKS COMPLETE.")


S1 rows: 1,732,544
Outer chunks: 174
Inner batch size: 100

    batch 001/100 | candidates=200,026 | accepted=277
    batch 002/100 | candidates=124,798 | accepted=317
    batch 003/100 | candidates=118,602 | accepted=323
    batch 004/100 | candidates=155,298 | accepted=289
    batch 005/100 | candidates=98,566 | accepted=347
    batch 006/100 | candidates=99,952 | accepted=457
    batch 007/100 | candidates=108,042 | accepted=294
    batch 008/100 | candidates=93,217 | accepted=340
    batch 009/100 | candidates=118,631 | accepted=309
    batch 010/100 | candidates=105,157 | accepted=302
    batch 011/100 | candidates=227,270 | accepted=360
    batch 012/100 | candidates=170,122 | accepted=272
    batch 013/100 | candidates=105,607 | accepted=310
    batch 014/100 | candidates=101,257 | accepted=312
    batch 015/100 | candidates=97,276 | accepted=248
    batch 016/100 | candidates=105,344 | accepted=343
    batch 017/100 | candidates=170,526 | accepted=315
    batch 018/100 | candid

KeyboardInterrupt: 

## 11. Combine accepted predictions

In [ ]:
# ============================================================
# STREAM CHUNKS INTO FINAL SUBMISSION
# ============================================================

chunk_files = sorted(CHUNK_DIR.glob("matches_*.tsv"))

if not chunk_files:
    raise FileNotFoundError("No inference chunk files found.")

if RESULT_PATH.exists():
    RESULT_PATH.unlink()

first = True
total_rows = 0

with RESULT_PATH.open("w", encoding="utf-8", newline="") as out:
    for path in chunk_files:
        df = pd.read_csv(path, sep="\t", dtype=str)

        if df.empty:
            del df
            continue

        # Keep only the two required columns.
        df = df[
            ["source1_entity_id", "target_entity_id"]
        ]

        df.to_csv(
            out,
            sep="\t",
            index=False,
            header=first,
        )

        first = False
        total_rows += len(df)

        del df
        gc.collect()

print("Final pair rows:", f"{total_rows:,}")
print("Submission pairs:", RESULT_PATH)


## 12. Prediction sanity checks

In [ ]:

if accepted_matches.duplicated(
    ["source1_entity_id","target_entity_id"]
).any():
    raise ValueError("Duplicate S1-target pairs found.")

if accepted_matches["target_entity_id"].str.startswith("S1-").any():
    raise ValueError("S1 ID appears as a target.")

    raise ValueError("Unknown target ID detected.")

print("Prediction checks passed.")


## 13. Write `matching_results.tsv`

In [ ]:

if accepted_matches.empty:
    grouped = pd.Series(dtype=str)
else:
    grouped = (
        accepted_matches
        .sort_values(["source1_entity_id","target_entity_id"])
        .groupby("source1_entity_id", sort=False)["target_entity_id"]
        .agg(",".join)
    )

matching_results = pd.DataFrame({
    "source1_entity_id": test_s1["entity_id"].astype(str)
})

matching_results["matched_entity_ids"] = (
    matching_results["source1_entity_id"]
    .map(grouped)
    .fillna("")
)

# Remove any accidental duplicate target IDs within a row.
matching_results["matched_entity_ids"] = matching_results[
    "matched_entity_ids"
].map(
    lambda x: ",".join(dict.fromkeys(x.split(","))) if x else ""
)

matching_results.to_csv(
    RESULT_PATH, sep="\t", index=False
)

print("Saved:", RESULT_PATH)
print("Rows:", len(matching_results))
print("Empty:", int((matching_results["matched_entity_ids"] == "").sum()))
print("Non-empty:", int((matching_results["matched_entity_ids"] != "").sum()))


## 14. Final structural validation

In [ ]:

if len(matching_results) != len(test_s1):
    raise ValueError("Wrong number of output rows.")

if set(matching_results["source1_entity_id"]) != set(test_s1["entity_id"]):
    raise ValueError("Output S1 IDs do not exactly match test S1 IDs.")

if matching_results["source1_entity_id"].duplicated().any():
    raise ValueError("Duplicate S1 rows.")

for value in matching_results.loc[
    matching_results["matched_entity_ids"] != "",
    "matched_entity_ids"
]:
    ids = value.split(",")
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate target inside prediction.")
    if any(x.startswith("S1-") for x in ids):
        raise ValueError("S1 target found.")

print("All structural checks passed.")


## 15. Official validator

The challenge documentation says to run `utils/validate_submission.py` before submission.

The exact CLI is repository-specific, so this cell first attempts the direct path form and prints the validator's output.


In [ ]:

validator = ROOT / "utils" / "validate_submission.py"

if not validator.exists():
    print("Validator not found:", validator)
else:
    proc = subprocess.run(
        [sys.executable, str(validator), str(RESULT_PATH)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    print("Return code:", proc.returncode)
    print(proc.stdout)
    print(proc.stderr)

    if proc.returncode != 0:
        print(
            "If the CLI expects different arguments, run:\n"
            f"  {sys.executable} {validator} --help"
        )


## 16. Final submission summary

In [ ]:

size_mb = RESULT_PATH.stat().st_size / (1024*1024)

print("=" * 60)
print("FINAL SUBMISSION")
print("=" * 60)
print("File:", RESULT_PATH)
print(f"Size: {size_mb:.2f} MB")
print("Rows:", len(matching_results))
print("Threshold:", MATCH_THRESHOLD)
print("Block threshold:", TOKEN_FREQ_THRESHOLD)
print("Block top-K:", TOP_K)
print("Accepted pairs:", len(accepted_matches))
print("Empty S1 predictions:",
      int((matching_results["matched_entity_ids"] == "").sum()))
print("\nFirst 10 rows:")
display(matching_results.head(10))


### Before leaderboard upload

The output file is:

```text
output/matching_results.tsv
```

Do not delete `output/inference_chunks/` until the result has passed validation.

If we later change the matcher threshold after proper validation, update `MATCH_THRESHOLD` and rerun inference. The chunk system will otherwise reuse old predictions, so delete the chunk files before rerunning with a different threshold.
